# Sales CI/CD workshop
Synthetic, embedded data; no uploads, downloads, secrets, or package installs.
Attach the current workspace's schema-enabled `lh_workshop` as the default lakehouse.
**Run all overwrites only `dbo.sales_wk` in that lakehouse. Review the binding first.**
Baseline: 3 rows, amount total 350.00. Tax release: also amount_with_tax total 385.00.


In [ ]:
from decimal import Decimal

RELEASE = "baseline"
SALES_ROWS = [("Online", "100.00"), ("Store", "200.00"), ("Partner", "50.00")]

def build_sales(release):
    if release not in ("baseline", "tax"):
        raise ValueError(f"Unknown release: {release}")
    rows = []
    for channel, amount in SALES_ROWS:
        amount = Decimal(amount)
        row = (channel, amount)
        if release == "tax":
            row += ((amount * Decimal("1.10")).quantize(Decimal("0.01")),)
        rows.append(row)
    return rows

sales_rows = build_sales(RELEASE)


In [ ]:
import notebookutils
from pyspark.sql import functions as F

context = notebookutils.runtime.context
workspace_id = context["currentWorkspaceId"]
lakehouse_workspace_id = context["defaultLakehouseWorkspaceId"]
lakehouse_name = context["defaultLakehouseName"]
lakehouse_id = context["defaultLakehouseId"]
if not lakehouse_id or lakehouse_workspace_id != workspace_id or lakehouse_name != "lh_workshop":
    raise RuntimeError("STOP: attach lh_workshop from THIS workspace as the default lakehouse, then restart the session.")
print(f"workspace={context['currentWorkspaceName']} ({workspace_id})")
print(f"lakehouse={lakehouse_name} ({lakehouse_id}); release={RELEASE}")

table_name = "lh_workshop.dbo.sales_wk"
schema = "channel string, amount decimal(12,2)"
if RELEASE == "tax":
    schema += ", amount_with_tax decimal(12,2)"
sales = spark.createDataFrame(sales_rows, schema=schema)
sales.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)


In [ ]:
saved = spark.table(table_name)
assert saved.count() == 3, "Expected exactly 3 persisted rows"
actual_rows = sorted(tuple(row) for row in saved.collect())
assert actual_rows == sorted(build_sales(RELEASE)), "Persisted rows differ from this release"
assert saved.agg(F.sum("amount")).first()[0] == Decimal("350.00"), "Wrong amount total"
if RELEASE == "tax":
    assert saved.agg(F.sum("amount_with_tax")).first()[0] == Decimal("385.00"), "Wrong tax-inclusive total"
else:
    assert saved.columns == ["channel", "amount"], "Unexpected baseline columns"
display(saved.orderBy("channel"))
print(f"PASS: release={RELEASE}; rows=3; amount=350.00" + ("; amount_with_tax=385.00" if RELEASE == "tax" else ""))
